# Comparativo ITT Zonas

Este notebook consolida los resultados generados por los notebooks individuales:

- ITT Roosevelt.
- ITT Avenida Ciudad de Cali.
- ITT Barrio Obrero.

La idea es leer los archivos exportados en `outputs/` y generar un resumen comparativo en `outputs/consolidado/`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

OUTPUTS = Path("../outputs")
CONSOLIDADO = OUTPUTS / "consolidado"
CONSOLIDADO.mkdir(parents=True, exist_ok=True)

zonas = {
    "ITT Roosevelt": OUTPUTS / "itt_roosevelt" / "itt_roosevelt_resumen_itt.csv",
    "ITT Avenida Ciudad de Cali": OUTPUTS / "itt_avenida_ciudad_de_cali" / "itt_avenida_ciudad_de_cali_resumen_itt.csv",
    "ITT Barrio Obrero": OUTPUTS / "itt_barrio_obrero" / "itt_barrio_obrero_resumen_itt.csv",
}

zonas

In [ ]:
# Cargar resultados disponibles

frames = []

for zona, path in zonas.items():
    if path.exists():
        df = pd.read_csv(path)
        df["zona"] = zona
        frames.append(df)
        print(f"Cargado: {zona} -> {path}")
    else:
        print(f"No encontrado: {zona} -> {path}")

if frames:
    df_consolidado = pd.concat(frames, ignore_index=True)
else:
    df_consolidado = pd.DataFrame(columns=["zona", "anio", "itt"])

df_consolidado

In [ ]:
# Ranking por ITT promedio

if not df_consolidado.empty and "itt" in df_consolidado.columns:
    ranking = (
        df_consolidado
        .groupby("zona", as_index=False)
        .agg(itt_promedio=("itt", "mean"))
        .sort_values("itt_promedio", ascending=False)
    )
else:
    ranking = pd.DataFrame(columns=["zona", "itt_promedio"])

ranking

In [ ]:
# Exportar consolidado

consolidado_excel = CONSOLIDADO / "comparativo_itt_zonas.xlsx"
consolidado_csv = CONSOLIDADO / "resumen_itt_zonas.csv"
ranking_csv = CONSOLIDADO / "ranking_itt_zonas.csv"

with pd.ExcelWriter(consolidado_excel, engine="openpyxl") as writer:
    df_consolidado.to_excel(writer, sheet_name="Resultados_Consolidados", index=False)
    ranking.to_excel(writer, sheet_name="Ranking_Zonas", index=False)

df_consolidado.to_csv(consolidado_csv, index=False, encoding="utf-8-sig")
ranking.to_csv(ranking_csv, index=False, encoding="utf-8-sig")

print("Archivos exportados:")
print("-", consolidado_excel)
print("-", consolidado_csv)
print("-", ranking_csv)

In [ ]:
# Gráfica simple de ranking, si hay datos disponibles

import matplotlib.pyplot as plt

if not ranking.empty:
    ax = ranking.plot(kind="bar", x="zona", y="itt_promedio", legend=False, figsize=(10, 5))
    ax.set_title("Ranking ITT promedio por zona")
    ax.set_xlabel("Zona")
    ax.set_ylabel("ITT promedio")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No hay datos para graficar todavía.")